# 01 — Data Cleaning and Feature Engineering

This notebook loads the Occuspace Rec Center export, applies cleaning rules, engineers time and academic-calendar features, assigns a time-based train/validation/test split, and saves `data/rec_center_clean.parquet` for downstream modeling.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if (ROOT / "rec_center_utils.py").exists():
    pass
elif (ROOT / "rec_center" / "rec_center_utils.py").exists():
    ROOT = ROOT / "rec_center"
elif (ROOT.parent / "rec_center_utils.py").exists():
    ROOT = ROOT.parent
else:
    raise FileNotFoundError("Could not locate rec_center_utils.py")
sys.path.insert(0, str(ROOT))

from rec_center_utils import (
    TRAIN_END,
    VAL_END,
    clean_data,
    data_path,
    figures_path,
    load_clean_data,
    load_raw_data,
    save_clean_data,
)

sns.set_theme(style="whitegrid", context="notebook")


c:\Users\tyler\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Load raw data

Source workbook: `datasets/rec_center_usage.xlsx` (~223k rows, six locations, 30-minute intervals).


In [2]:

raw = load_raw_data()
raw.head()



,location,timestamp,date,day_of_week,week_of_year,time,hour,average_occupancy,average_utilization,peak_occupancy,peak_utilization,capacity,location_path
0,Lower Exercise Room,2023-05-13 06:00:00,2023-05-13,7,19,06:00:00,6,2,0.02,3,0.04,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...
1,Lower Exercise Room,2023-05-13 06:30:00,2023-05-13,7,19,06:30:00,6,2,0.02,3,0.04,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...
2,Lower Exercise Room,2023-05-13 07:00:00,2023-05-13,7,19,07:00:00,7,2,0.02,3,0.04,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...
3,Lower Exercise Room,2023-05-13 07:30:00,2023-05-13,7,19,07:30:00,7,3,0.04,3,0.04,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...
4,Lower Exercise Room,2023-05-13 08:00:00,2023-05-13,7,19,08:00:00,8,10,0.12,15,0.18,85,Cal Poly > Rec Center > 1st Floor > Lower Exe...


## Cleaning decisions

- Drop duplicate `location` + `timestamp` pairs (keep first after sorting).
- Cap `average_utilization` at 1.0 for modeling while retaining the raw value.
- Exclude peak occupancy/utilization from predictors in modeling notebooks to avoid same-interval leakage.
- Engineer cyclical time features and Cal Poly academic calendar flags.


In [3]:

cleaned, report = clean_data(raw)
report



{'rows_raw': 223499,
 'duplicate_location_timestamp': 216,
 'utilization_above_one': 22069,
 'rows_clean': 223283,
 'split_counts': {'test': 101008, 'train': 82679, 'validation': 39596}}

In [4]:

cleaned[["location", "timestamp", "average_utilization", "average_utilization_raw", "usage_level", "split"]].head()



,location,timestamp,average_utilization,average_utilization_raw,usage_level,split
38059,1st Floor,2023-05-13 06:00:00,0.02,0.02,low,train
38060,1st Floor,2023-05-13 06:30:00,0.02,0.02,low,train
38061,1st Floor,2023-05-13 07:00:00,0.02,0.02,low,train
38062,1st Floor,2023-05-13 07:30:00,0.04,0.04,low,train
38063,1st Floor,2023-05-13 08:00:00,0.12,0.12,low,train


## Time-based split

- **Train:** through 2024-06-30
- **Validation:** through 2024-12-31
- **Test:** remaining recent months (held out for final evaluation)


In [5]:

cleaned["split"].value_counts()



split
test          101008
train          82679
validation     39596
Name: count, dtype: int64

In [6]:

output_path = save_clean_data(cleaned)
output_path



WindowsPath('C:/Users/tyler/OneDrive/Desktop/Advanced Machine Learning/rec_center/data/rec_center_clean.parquet')